# 06 - Data Transfer Smoke Ingestion

Checkpoint 6: inspect saved real Data Transfer smoke-ingestion artifacts. This notebook reads local outputs only. It does not submit jobs, authenticate, call Fink services, or fetch cutouts/FITS/images.

Interpretation guardrail: the current delivery is `in_tns`-filtered light static packet data. It proves smoke transfer and ingestion, not full-night all-alert completeness.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SMOKE_ROOT = PROJECT_ROOT / "outputs/data_transfer/smoke_delivery"

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None

latest = read_json(SMOKE_ROOT / "latest_run.json")
latest

In [ ]:
run_id = latest["run_id"] if latest else sorted(p.name for p in SMOKE_ROOT.iterdir() if p.is_dir())[-1]
run_dir = SMOKE_ROOT / run_id
processed_dir = Path(latest["processed_dir"]) if latest else PROJECT_ROOT / "data/processed/data_transfer/smoke_delivery" / run_id

manifest = read_json(processed_dir / "manifest.json")
inventory = read_json(run_dir / "raw_field_inventory.json")
nested = read_json(run_dir / "nested_conversion_report.json") or read_json(processed_dir / "nested_conversion_report.json")
validation = read_json(run_dir / "validation_report.json")
diagnostics = read_json(run_dir / "smoke_diagnostics.json")
next_action = read_json(PROJECT_ROOT / "outputs/data_transfer/data_transfer_next_action.json")

display(Markdown(f"## Run `{run_id}`"))
display(pd.DataFrame([{
    "topic": manifest.get("topic") if manifest else None,
    "raw_files": manifest.get("raw_file_count") if manifest else None,
    "raw_rows": manifest.get("raw_row_count") if manifest else None,
    "scope": manifest.get("scope") if manifest else None,
    "full_night_complete": manifest.get("full_night_complete") if manifest else None,
    "validation_status": manifest.get("validation_status") if manifest else None,
}]))

In [ ]:
display(Markdown("## Raw Field Inventory"))
display(pd.DataFrame({
    "metric": ["file_count", "total_rows", "schema_group_count", "nested_columns"],
    "value": [
        inventory.get("file_count") if inventory else None,
        inventory.get("total_rows") if inventory else None,
        inventory.get("schema_group_count") if inventory else None,
        ", ".join(inventory.get("nested_columns", [])) if inventory else None,
    ],
}))

display(Markdown("## Processed Table Shapes"))
display(pd.DataFrame.from_dict(manifest.get("table_shapes", {}), orient="index") if manifest else pd.DataFrame())

display(Markdown("## Nested Conversion"))
converted = nested.get("converted_columns", {}) if nested else {}
display(pd.DataFrame([{"table": key, "converted_columns": ", ".join(value)} for key, value in converted.items()]))

In [ ]:
display(Markdown("## Validation"))
checks = validation.get("checks", []) if validation else []
display(pd.DataFrame([
    {
        "severity": item.get("severity"),
        "check": item.get("check"),
        "passed": item.get("passed"),
        "message": item.get("message"),
    }
    for item in checks
]))

display(Markdown("## Diagnostics"))
if diagnostics:
    display(pd.DataFrame([
        {"metric": "alert_rows", "value": diagnostics.get("alerts", {}).get("rows")},
        {"metric": "unique_objects", "value": diagnostics.get("objects", {}).get("unique_objects")},
        {"metric": "unique_sources", "value": diagnostics.get("objects", {}).get("unique_sources")},
        {"metric": "lc_feature_rows", "value": diagnostics.get("lightcurve_features", {}).get("rows")},
    ]))
    display(pd.DataFrame(diagnostics.get("promising_first_notebook_fields", []), columns=["promising_field"]))

In [ ]:
display(Markdown("## Figures"))
figures = diagnostics.get("figures", []) if diagnostics else []
for item in figures:
    path = Path(item.get("path", ""))
    if path.exists():
        display(Markdown(f"### `{item.get('name')}`"))
        display(Image(filename=str(path)))

display(Markdown("## Readiness Decision"))
display(pd.DataFrame([next_action or {}]))

## Final Interpretation

- Data Transfer access works for the saved smoke topic.
- The real smoke delivery ingests into processed Parquet tables after nested-field sanitization.
- Raw responses remain preserved before normalization under `data/raw/data_transfer/`.
- `lc_features` is preserved as JSON and expanded when possible.
- The delivery is tag-filtered and not full-night complete.
- The pipeline can proceed toward a first full-night request only after explicit user confirmation.